In [1]:
# train_titanic_model.py

import seaborn as sns
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
import joblib

# 1. Load data
df = sns.load_dataset("titanic")[[
    "pclass","sex","age","sibsp","parch","fare","embarked","survived"
]].dropna(subset=["survived"])

# 2. Feature / target split
X = df.drop("survived", axis=1)
y = df["survived"]

# 3. Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 4. Preprocessing pipelines
numeric_features = ["age", "sibsp", "parch", "fare"]
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_features = ["pclass", "sex", "embarked"]
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features),
])

# 5. Full pipeline: preprocessing + classifier
clf = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=500, random_state=42
    ))
])

# 6. Train
clf.fit(X_train, y_train)

# 7. Evaluate (optional)
acc = clf.score(X_test, y_test)
print(f"Test accuracy: {acc:.3f}")

# 8. Serialize pipeline
joblib.dump(clf, "titanic_pipeline.joblib")
print("Saved pipeline to titanic_pipeline.joblib")


Test accuracy: 0.810
Saved pipeline to titanic_pipeline.joblib


In [5]:
# model.py

import joblib
import pandas as pd
import numpy as np

# Load the serialized scikit-learn pipeline
_pipeline = joblib.load("titanic_pipeline.joblib")

def predict_passenger(data: dict) -> dict:
    """
    data keys must be exactly:
      ["pclass","sex","age","sibsp","parch","fare","embarked"]
    Returns a dict: {"survived": bool, "probability": float}
    """
    # 1) Define the feature order
    features = ["pclass","sex","age","sibsp","parch","fare","embarked"]
    # 2) Build a single-row DataFrame
    df = pd.DataFrame([data], columns=features)
    # 3) Ask the pipeline for probabilities
    proba = _pipeline.predict_proba(df)[0][1]
    survived = bool(proba >= 0.5)
    return {"survived": survived, "probability": round(proba, 3)}

In [6]:
X_test.iloc[1].values

array([np.int64(3), 'male', np.float64(44.0), np.int64(0), np.int64(1),
       np.float64(16.1), 'S'], dtype=object)

In [7]:
predict_passenger({"pclass":3,"sex":"male","age":44.0,"sibsp":0,
                   "parch":1,"fare":16.1,"embarked":'S'})

{'survived': False, 'probability': np.float64(0.151)}